# ⏱️ ESG Time Series Forecasting
**Objective:** Forecast GrowthRate, Revenue, and ESG_Overall for 2026–2030.

**Methods:** SARIMA (statsmodels) & Prophet (Facebook)

**Evaluation:** MAPE and RMSE

---

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# SARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Prophet
from prophet import Prophet

# Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

import os
os.makedirs('outputs', exist_ok=True)

print('Libraries loaded successfully!')

## 2. Load & Prepare Time Series Data

In [ ]:
try:
 df = pd.read_csv('esg_processed.csv')
 print('Loaded esg_processed.csv')
except FileNotFoundError:
 df = pd.read_csv('raw_data.csv')
 print('Loaded raw_data.csv')

print(f'Shape: {df.shape}')
print(f'Year range: {df["Year"].min()} - {df["Year"].max()}')
df.head()

In [ ]:
# Aggregate to annual mean across all companies
METRICS = ['GrowthRate', 'Revenue', 'ESG_Overall']
ts_df = df.groupby('Year')[METRICS].mean().reset_index()
ts_df = ts_df.sort_values('Year').reset_index(drop=True)

print('Annual aggregated time series:')
ts_df

In [ ]:
fig = make_subplots(rows=3, cols=1, subplot_titles=METRICS, shared_xaxes=True)
colors = ['#3498db', '#2ecc71', '#e67e22']

for i, (metric, color) in enumerate(zip(METRICS, colors), 1):
 fig.add_trace(go.Scatter(x=ts_df['Year'], y=ts_df[metric],
 mode='lines+markers', name=metric, line=dict(color=color, width=2)),
 row=i, col=1)

fig.update_layout(title='Historical Trends (2015–2025)', height=600, showlegend=True)
fig.show()

## 3. Stationarity Check (ADF Test)

In [ ]:
def adf_test(series, name):
 result = adfuller(series.dropna())
 print(f'{name}:')
 print(f' ADF Statistic : {result[0]:.4f}')
 print(f' p-value : {result[1]:.4f}')
 print(f' Stationary : {"Yes" if result[1] < 0.05 else "No (differencing needed)"}')
 print()

for metric in METRICS:
 adf_test(ts_df[metric], metric)

In [ ]:
fig, axes = plt.subplots(len(METRICS), 2, figsize=(14, 10))

for i, metric in enumerate(METRICS):
 series = ts_df[metric].dropna()
 plot_acf(series, ax=axes[i, 0], lags=min(8, len(series)//2 - 1), title=f'ACF - {metric}')
 plot_pacf(series, ax=axes[i, 1], lags=min(8, len(series)//2 - 1), title=f'PACF - {metric}')

plt.tight_layout()
plt.savefig('outputs/acf_pacf_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Train-Test Split for Evaluation

In [ ]:
# Use 2015-2022 for training, 2023-2025 for testing
TRAIN_END = 2022
TEST_START = 2023
FORECAST_YEARS = list(range(2026, 2031))

ts_train = ts_df[ts_df['Year'] <= TRAIN_END].copy()
ts_test = ts_df[ts_df['Year'] >= TEST_START].copy()

print(f'Train: {ts_train["Year"].min()} - {ts_train["Year"].max()} ({len(ts_train)} years)')
print(f'Test : {ts_test["Year"].min()} - {ts_test["Year"].max()} ({len(ts_test)} years)')
print(f'Forecast: {FORECAST_YEARS}')

## 5. SARIMA Forecasting

In [ ]:
sarima_results = {}
sarima_forecasts = {}

# SARIMA order — annual data, no strong seasonality (s=1)
SARIMA_ORDER = (1, 1, 1)
SARIMA_SEASONAL = (0, 0, 0, 1)

for metric in METRICS:
 print(f'Fitting SARIMA for {metric}...')
 train_series = ts_train[metric].values
 test_series = ts_test[metric].values
 n_test = len(test_series)

 model = SARIMAX(train_series, order=SARIMA_ORDER,
 seasonal_order=SARIMA_SEASONAL,
 enforce_stationarity=False,
 enforce_invertibility=False)
 fit = model.fit(disp=False)

 # Predict on test set
 test_pred = fit.forecast(steps=n_test)

 # Forecast future (2026-2030) — refit on full data
 full_model = SARIMAX(ts_df[metric].values, order=SARIMA_ORDER,
 seasonal_order=SARIMA_SEASONAL,
 enforce_stationarity=False,
 enforce_invertibility=False)
 full_fit = full_model.fit(disp=False)
 future_pred = full_fit.forecast(steps=5)

 rmse = np.sqrt(mean_squared_error(test_series, test_pred))
 mape = mean_absolute_percentage_error(test_series, test_pred) * 100

 sarima_results[metric] = {'RMSE': rmse, 'MAPE': mape}
 sarima_forecasts[metric] = future_pred
 print(f' RMSE={rmse:.4f} | MAPE={mape:.2f}%')

print('\nSARIMA complete.')

## 6. Prophet Forecasting

In [ ]:
prophet_results = {}
prophet_forecasts = {}

for metric in METRICS:
 print(f'Fitting Prophet for {metric}...')
 # Prophet requires columns 'ds' and 'y'
 prophet_df = ts_df[['Year', metric]].rename(columns={'Year': 'ds', metric: 'y'})
 prophet_df['ds'] = pd.to_datetime(prophet_df['ds'], format='%Y')

 train_df = prophet_df[prophet_df['ds'].dt.year <= TRAIN_END]
 test_df = prophet_df[prophet_df['ds'].dt.year >= TEST_START]

 m = Prophet(yearly_seasonality=False, weekly_seasonality=False,
 daily_seasonality=False, interval_width=0.95)
 m.fit(train_df)

 # Test period predictions
 future_test = m.make_future_dataframe(periods=len(test_df), freq='YS')
 forecast_test = m.predict(future_test)
 test_pred = forecast_test.tail(len(test_df))['yhat'].values
 test_actual = test_df['y'].values

 # Future forecast on full data
 m_full = Prophet(yearly_seasonality=False, weekly_seasonality=False,
 daily_seasonality=False, interval_width=0.95)
 m_full.fit(prophet_df)
 future_full = m_full.make_future_dataframe(periods=5, freq='YS')
 forecast_full = m_full.predict(future_full)
 future_only = forecast_full.tail(5)

 rmse = np.sqrt(mean_squared_error(test_actual, test_pred))
 mape = mean_absolute_percentage_error(test_actual, test_pred) * 100

 prophet_results[metric] = {'RMSE': rmse, 'MAPE': mape}
 prophet_forecasts[metric] = {
 'yhat': future_only['yhat'].values,
 'yhat_lower': future_only['yhat_lower'].values,
 'yhat_upper': future_only['yhat_upper'].values
 }
 print(f' RMSE={rmse:.4f} | MAPE={mape:.2f}%')

print('\nProphet complete.')

## 7. Model Comparison & Forecast Visualization

In [ ]:
comparison_rows = []
for metric in METRICS:
 comparison_rows.append({
 'Metric': metric,
 'SARIMA_RMSE': sarima_results[metric]['RMSE'],
 'SARIMA_MAPE': sarima_results[metric]['MAPE'],
 'Prophet_RMSE': prophet_results[metric]['RMSE'],
 'Prophet_MAPE': prophet_results[metric]['MAPE'],
 })

comparison_df = pd.DataFrame(comparison_rows)
print('=== SARIMA vs Prophet Comparison ===')
comparison_df

In [ ]:
for metric in METRICS:
 fig = go.Figure()

 # Historical
 fig.add_trace(go.Scatter(x=ts_df['Year'], y=ts_df[metric],
 mode='lines+markers', name='Historical',
 line=dict(color='#2c3e50', width=2)))

 # SARIMA Forecast
 fig.add_trace(go.Scatter(x=FORECAST_YEARS, y=sarima_forecasts[metric],
 mode='lines+markers', name='SARIMA Forecast',
 line=dict(color='#e74c3c', dash='dash', width=2)))

 # Prophet Forecast + CI
 pf = prophet_forecasts[metric]
 fig.add_trace(go.Scatter(x=FORECAST_YEARS, y=pf['yhat'],
 mode='lines+markers', name='Prophet Forecast',
 line=dict(color='#3498db', dash='dot', width=2)))
 fig.add_trace(go.Scatter(
 x=FORECAST_YEARS + FORECAST_YEARS[::-1],
 y=list(pf['yhat_upper']) + list(pf['yhat_lower'][::-1]),
 fill='toself', fillcolor='rgba(52,152,219,0.15)',
 line=dict(color='rgba(255,255,255,0)'),
 name='Prophet 95% CI'))

 fig.add_vline(x=2025.5, line_dash='dash', line_color='gray',
 annotation_text='Forecast Start')

 fig.update_layout(
 title=f'{metric} Forecast 2026–2030',
 xaxis_title='Year', yaxis_title=metric,
 legend=dict(orientation='h', yanchor='bottom', y=1.02),
 height=450
 )
 fig.show()
 fig.write_html(f'outputs/forecast_{metric.lower()}.html')

print('Forecast plots saved to outputs/')

## 8. Save Forecast Results

In [ ]:
forecast_rows = []
for year_idx, year in enumerate(FORECAST_YEARS):
 row = {'Year': year}
 for metric in METRICS:
 row[f'{metric}_SARIMA'] = sarima_forecasts[metric][year_idx]
 row[f'{metric}_Prophet'] = prophet_forecasts[metric]['yhat'][year_idx]
 row[f'{metric}_Prophet_Lower'] = prophet_forecasts[metric]['yhat_lower'][year_idx]
 row[f'{metric}_Prophet_Upper'] = prophet_forecasts[metric]['yhat_upper'][year_idx]
 forecast_rows.append(row)

forecast_df = pd.DataFrame(forecast_rows)
forecast_df.to_csv('outputs/forecast_results_2026_2030.csv', index=False)
comparison_df.to_csv('outputs/model_comparison_timeseries.csv', index=False)

print('Saved:')
print(' - outputs/forecast_results_2026_2030.csv')
print(' - outputs/model_comparison_timeseries.csv')
print('\n=== Final Forecast Table ===')
forecast_df

## 9. Key Findings

In [ ]:
print('=== KEY FINDINGS ===')
for metric in METRICS:
 s = sarima_results[metric]
 p = prophet_results[metric]
 best = 'SARIMA' if s['MAPE'] < p['MAPE'] else 'Prophet'
 print(f'{metric}:')
 print(f' Best Model : {best}')
 print(f' SARIMA MAPE : {s["MAPE"]:.2f}%')
 print(f' Prophet MAPE: {p["MAPE"]:.2f}%')
 print()